# RLHF: katta LLM hakam (API) + RLOO, SFT v3 LoRA (Qwen2.5-1.5B) ustida

Reward model o'rniga **katta LLM** har bir javobni RL paytida jonli baholaydi (reward model jadvalidagi 5-variant).
Standart: **Google Gemini** `gemini-3.5-flash-lite` (bepul tarif). OpenAI'ga mos API'si bor istalgan provayder
ishlaydi (Groq, OpenAI, OpenRouter, Mistral...): faqat `JUDGE_BASE_URL` va `JUDGE_MODEL` o'zgaradi.

Hakamlar bizning 40 ta val juftlikda (lokal test, `rlhf/eval_judge.py`):

| Hakam | Aniqlik | Teng ball |
|---|---|---|
| **Gemini 3.5 Flash-Lite** | **0.90** | 0 |
| Groq `openai/gpt-oss-120b` | 0.82 | 5 |
| Bizning v2 reward model (181 juftlik) | 0.84 | — |

`gemini-3.5-flash` (Lite emas) bepul tarifda kuniga atigi **20 ta so'rov** beradi — bizga ~440 kerak.

- Bitta so'rov = bitta prompt'ning **4 ta javobi birga** (tartibi aralashtiriladi): 776 prompt = 776 so'rov.
- So'rov xato bo'lsa, o'sha prompt'ning 4 ta javobiga bir xil ball beriladi: RLOO undan hech narsa o'rganmaydi.

**Boshlashdan oldin:**

1. **API kalit:** https://aistudio.google.com/apikey -> **Create API key** (bepul, Google akkaunt kifoya).
2. Kaggle'da **Add-ons -> Secrets -> Add Secret**: nomi **`JUDGE_API_KEY`**, qiymati — kalit. **Attach** belgisini yoqing.
   Kalitni notebook ichiga yozmang: notebook saqlanganda u boshqalarga ko'rinishi mumkin.
3. **GPU T4 x2**, **Internet -> On**, **Add Input** -> `sft-v3-adapters`.
4. **Save Version -> Save & Run All (Commit)**.

**Bepul limit:** bepul tariflarda kunlik so'rov limiti bor. Shuning uchun quyida `LIMIT_PROMPTS = 400`
(400 so'rov). Pulli tarifda `None` qiling (776 ta). Ko'p `hakam xatosi` chiqsa — limit tugagan.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")
!ls -lh Qwen2.5-1.5B/*.safetensors

## 3. SFT v3 LoRA adapter (policy va reference shundan boshlanadi)

In [ ]:
import glob, os, shutil, subprocess, zipfile
target = "chat_ui/adapters/sft-lora-v3"
found = glob.glob("/kaggle/input/**/sft-lora-v3/adapter_config.json", recursive=True)
if not found:   # zip ochilmagan holda yuklangan bo'lsa
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            names = [n for n in f.namelist() if "sft-lora-v3/" in n]
            if names:
                f.extractall("/kaggle/working/unzipped", members=names)
    found = glob.glob("/kaggle/working/unzipped/**/sft-lora-v3/adapter_config.json", recursive=True)
if found:
    shutil.copytree(os.path.dirname(found[0]), target, dirs_exist_ok=True)
    print("SFT v3 adapter topildi:", os.path.dirname(found[0]))
else:
    print("Adapter topilmadi: SFT v3 LoRA qaytadan o'qitiladi...")
    subprocess.run(["python", "sft_v2/train.py", "--method", "lora", "--data", "sft_v3/data",
                    "--name", "sft-lora-v3", "--epochs", "3", "--rank", "64"], check=True)
print(os.listdir(target))

## 4. API sozlamalari

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["JUDGE_API_KEY"] = UserSecretsClient().get_secret("JUDGE_API_KEY")
os.environ["JUDGE_BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"
os.environ["JUDGE_MODEL"] = "gemini-3.5-flash-lite"
# Groq:   JUDGE_BASE_URL = "https://api.groq.com/openai/v1",  JUDGE_MODEL = "openai/gpt-oss-120b"
# OpenAI: JUDGE_BASE_URL = "https://api.openai.com/v1",       JUDGE_MODEL = "gpt-..."
LIMIT_PROMPTS = 400   # bepul limit uchun; None = barcha 776 prompt
LIMIT_FLAG = f"--limit-prompts {LIMIT_PROMPTS}" if LIMIT_PROMPTS else ""
print("Hakam:", os.environ["JUDGE_MODEL"], "|", os.environ["JUDGE_BASE_URL"])

## 5. Hakamni tekshirish: 40 ta val juftlik (40 so'rov)

Bizning v2 reward model 181 juftlikda: **0.84**. 40 juftlik — taxminiy natija, lekin limitni tejaydi.

In [ ]:
!python rlhf/eval_judge.py --judge api --limit 40

## 6. RLOO (LLM hakam)

Har qadamda 8 ta prompt -> 8 ta parallel API so'rov. `hakam xatosi` satrlari ko'p chiqsa, limitga yetilgan.

In [ ]:
!python rlhf/train_ppo.py --algo rloo --group-size 4 --batch-size 8 --mini-batch 4 --kl-coef 0.1 \
    --judge api \
    --name sft-lora-rlhf-llm --rollouts rlhf_llm_rollouts.jsonl {LIMIT_FLAG}
import os
assert os.path.exists("chat_ui/adapters/sft-lora-rlhf-llm/adapter_config.json"), 'RLOO adapterni saqlamadi: yuqoridagi xatoni qarang'

In [ ]:
!rm -f /kaggle/working/sft-lora-rlhf-llm_results.zip
!zip -q -r /kaggle/working/sft-lora-rlhf-llm_results.zip chat_ui/adapters/sft-lora-rlhf-llm rlhf/logs/rlhf_llm_rollouts.jsonl
!ls -lh /kaggle/working/sft-lora-rlhf-llm_results.zip

## Baholash: 68 ta test savoli

SFT v3: 41/68. DPO v2: 44/68. RLHF v1: 40/68.

In [ ]:
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-rlhf-llm | tail -12
!zip -q /kaggle/working/sft-lora-rlhf-llm_results.zip eval/results/sft-lora-rlhf-llm.json
assert os.path.exists("eval/results/sft-lora-rlhf-llm.json"), 'Baholash ishlamadi: yuqoridagi xatoni qarang'

## Yuklab olish

**Output** -> `sft-lora-rlhf-llm_results.zip` -> **Download**. Loyiha ildizida oching (ichidagi yo'llar `chat_ui/...`,
`rlhf/...`, `eval/...` dan boshlanadi), keyin `python eval/summarize.py`.